# Giga Meter — install scope: is the app installed for all users, or only one?

**Why this notebook.** The Windows Setup Wizard asks whether to install Giga Meter for
**"Anyone who uses this computer (all users)"** or only for the account running the
installer. The installation guide tells deployers to pick all-users
(`giga-meter-docs · installation/installation-guide.md`, step 2) — because a school PC is
shared, and a per-user install only measures when that one Windows account is logged in.
Nobody has checked whether the field actually does it.

The measurements table records where the app runs from, so the choice is observable:

| `installed_path` | What it means |
|---|---|
| `C:\Program Files\...\Giga Meter\resources\app.asar` | **all-users** install — available to every Windows account |
| `C:\Users\<name>\AppData\Local\Programs\Giga Meter\...` | **per-user** install — only `<name>` runs it |
| anything else (`Desktop\`, `D:\`, ...) | **other** — unpacked/portable copy, not a supported install |

Two companion fields make it a real analysis rather than a path-string count:
`windows_username` (who is logged in when the test runs) and `device_hardware_id` (the
machine). Together they answer the question behind the question — *does choosing
all-users actually produce measurement from more than one account?*

**Scope and caveats, up front.**

* **Version gate.** `installed_path` and `windows_username` ship from **app 2.0.2**
  (Nov 2025) and are `NULL` on every earlier row *by design*. Compute every rate against
  a 2.0.2+ denominator or you will understate it badly — the same trap as the geolocation
  field. §2 states the covered subset before anything else.
* **This is the Windows fleet only.** Chrome-extension rows have no installed path, and
  the Windows fleet lives **only in `gigameter_production_db`** — staging holds the pilot
  clients and is a non-sample here.
* **`device_hardware_id` is deliberately scrubbed** for low-end hardware in Uzbekistan
  and Malawi (non-unique hardware IDs), so any per-*machine* statistic silently drops
  most of UZB. §4 reports its own denominator.
* **`browser_id` is an install, not a device** — reinstalls and reimages mint new IDs.
  "Installs" is the honest noun throughout.
* **Cohort mix dominates naive comparisons.** All-users installs are newer than per-user
  ones, so any lifetime/retention contrast between the two is a cohort artefact unless it
  is computed *within* a first-seen month. §6 shows the trap and controls for it.

*Related rules: R-17 (version-gated fields need a filtered denominator), R-18 (device
identity is layered and imperfect), R-22 (report the eligible denominator).*

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd() / "helpers"))
from load_measurements import get_trino_cursor
try:
    import giga_chart_style
    from giga_chart_style import GIGA_PRIMARY, GIGA_GREY, GIGA_GOOD, GIGA_MODERATE, GIGA_BAD
except Exception:
    GIGA_PRIMARY = {k: c for k, c in zip([300,400,600,700,800],
                    ['#a9caff','#7eb0ff','#277aff','#0050e6','#002d9c'])}
    GIGA_GREY = {300:'#dfdfdf',500:'#989898',600:'#6f6f6f',700:'#525252'}
    GIGA_GOOD, GIGA_MODERATE, GIGA_BAD = '#00d661', '#ffc93d', '#ed1c24'

# ── Parameters ───────────────────────────────────────────────────────────────
SINCE          = "2026-01-01"   # analysis window start (the pull is windowed)
COUNTRY        = None           # ISO3 for the §7 drill-down; None = largest deployment
MIN_INSTALLS   = 25             # country/school reporting floor (working figure, not a standard)
MULTI_ACCOUNT  = 5              # a path-user name on >= this many schools = shared/vendor account
REFRESH        = False          # True to re-query Trino instead of reading the cache

CACHE = Path.cwd() / "cache"; CACHE.mkdir(exist_ok=True)
OUT   = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)

SCOPE_COLOR = {"all_users": GIGA_GOOD, "per_user": GIGA_MODERATE, "other": GIGA_BAD}
SCOPE_ORDER = ["all_users", "per_user", "other"]

ISO2_TO_ISO3, ISO3_NAME = {}, {}
try:
    _ref = pd.read_json(Path.cwd() / "helpers" / "country_reference.json").T
    ISO2_TO_ISO3 = {v["iso2"]: k for k, v in _ref.iterrows()}
    ISO3_NAME    = {k: v["name"] for k, v in _ref.iterrows()}
except Exception as e:
    print(f"country_reference.json not read ({e}) — country_code stays iso2")

cur = get_trino_cursor()

def q(sql: str) -> pd.DataFrame:
    """Run a Trino query, return a DataFrame."""
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

MEAS = "gigameter_production_db.public.measurements"
print(f"window: {SINCE} → now · source: {MEAS}")

## §1 · Pull — one row per install, aggregated server-side

Two aggregates, both computed in Trino and cached:

* `COV` — rows and installs per `app_version` / country, **with and without** a path.
  This is what makes the version gate measurable (§2).
* `INST` — one row per `browser_id`: its scope, the Windows account it runs under, the
  machine, the school, and its measurement lifetime. ~30k rows; everything after this
  cell is pandas.

Where an install changed path mid-life (upgrade, reinstall under a different account) the
*latest* value wins via `max_by(..., timestamp)`, and `n_scopes` / `n_paths` record that
it moved — §6 uses those to separate a genuine re-install from a stable one.

In [ ]:
COV_CACHE  = CACHE / f"installscope_coverage_{SINCE}.parquet"
INST_CACHE = CACHE / f"installscope_installs_{SINCE}.parquet"

SCOPE_SQL = r"""CASE
             WHEN installed_path LIKE 'C:\Program Files%'                  THEN 'all_users'
             WHEN lower(installed_path) LIKE 'c:\users\%appdata\local%'    THEN 'per_user'
             ELSE 'other' END"""

if REFRESH or not COV_CACHE.exists():
    COV = q(rf"""
    SELECT  app_version
          , country_code
          , count(*)                                                                  AS rows_all
          , count(installed_path)                                                      AS rows_with_path
          , count(DISTINCT browser_id)                                                 AS installs_all
          , count(DISTINCT CASE WHEN installed_path  IS NOT NULL THEN browser_id END)  AS installs_with_path
          , count(DISTINCT CASE WHEN windows_username IS NOT NULL THEN browser_id END) AS installs_with_user
    FROM {MEAS}
    WHERE timestamp >= TIMESTAMP '{SINCE}' AND timestamp < current_timestamp
    GROUP BY 1, 2
    """)
    COV.to_parquet(COV_CACHE, index=False)
else:
    COV = pd.read_parquet(COV_CACHE)

if REFRESH or not INST_CACHE.exists():
    INST = q(rf"""
    WITH base AS (
      SELECT  browser_id, country_code, school_id, giga_id_school, device_hardware_id
            , windows_username, app_version, timestamp, installed_path
            , {SCOPE_SQL} AS scope
            , CASE WHEN lower(installed_path) LIKE 'c:\users\%'
                   THEN lower(split_part(installed_path, '\', 3)) END AS path_user
      FROM {MEAS}
      WHERE installed_path IS NOT NULL
        AND timestamp >= TIMESTAMP '{SINCE}' AND timestamp < current_timestamp
    )
    SELECT  browser_id
          , max_by(country_code,       timestamp) AS country_code
          , max_by(school_id,          timestamp) AS school_id
          , max_by(giga_id_school,     timestamp) AS giga_id_school
          , max_by(device_hardware_id, timestamp) AS device_hardware_id
          , max_by(windows_username,   timestamp) AS windows_username
          , max_by(app_version,        timestamp) AS app_version
          , max_by(scope,              timestamp) AS scope
          , max_by(path_user,          timestamp) AS path_user
          , max_by(installed_path,     timestamp) AS installed_path
          , count(DISTINCT scope)                 AS n_scopes
          , count(DISTINCT installed_path)        AS n_paths
          , count(DISTINCT school_id)             AS n_schools
          , count(DISTINCT windows_username)      AS n_usernames
          , count(*)                              AS n_tests
          , count(DISTINCT date(timestamp))       AS days_measured
          , min(timestamp)                        AS first_seen
          , max(timestamp)                        AS last_seen
    FROM base
    GROUP BY browser_id
    """)
    INST.to_parquet(INST_CACHE, index=False)
else:
    INST = pd.read_parquet(INST_CACHE)

DEV_CACHE = CACHE / f"installscope_devices_{SINCE}.parquet"
if REFRESH or not DEV_CACHE.exists():
    # Per MACHINE, counted over measurements — an install-level count would collapse the
    # case of one browser_id serving two logins, which is exactly what we are testing for.
    DEV = q(rf"""
    WITH base AS (
      SELECT  device_hardware_id, country_code, windows_username, browser_id, school_id
            , {SCOPE_SQL} AS scope
      FROM {MEAS}
      WHERE installed_path IS NOT NULL AND device_hardware_id IS NOT NULL
        AND windows_username IS NOT NULL
        AND timestamp >= TIMESTAMP '{SINCE}' AND timestamp < current_timestamp
    )
    SELECT  device_hardware_id
          , arbitrary(country_code)           AS country_code
          , arbitrary(scope)                  AS scope
          , count(DISTINCT scope)             AS n_scopes
          , count(DISTINCT windows_username)  AS n_accounts
          , count(DISTINCT browser_id)        AS n_installs
          , count(DISTINCT school_id)         AS n_schools
          , count(*)                          AS n_tests
    FROM base
    GROUP BY device_hardware_id
    """)
    DEV.to_parquet(DEV_CACHE, index=False)
else:
    DEV = pd.read_parquet(DEV_CACHE)

for df in (COV, INST, DEV):
    df["iso3"] = df["country_code"].map(ISO2_TO_ISO3).fillna(df["country_code"])
INST["cohort"]  = (pd.to_datetime(INST["first_seen"], utc=True).dt.tz_localize(None)
                     .dt.to_period("M").dt.to_timestamp())
INST["span_d"]  = (pd.to_datetime(INST["last_seen"], utc=True)
                   - pd.to_datetime(INST["first_seen"], utc=True)).dt.days
INST["moved"]   = INST["n_paths"] > 1

print(f"COV : {len(COV):,} rows · {COV.rows_all.sum():,} measurements")
print(f"INST: {len(INST):,} installs · {INST.n_tests.sum():,} measurements "
      f"· {INST.iso3.nunique()} countries · {INST.school_id.nunique():,} schools")
print(f"DEV : {len(DEV):,} machines reporting a hardware id")
display(INST.head(3))

## §2 · The version gate — what fraction of the fleet can even be judged

`installed_path` is written by the client, so a row without one is not a missing value:
it is a client too old to report. Before any percentage, this is the denominator.

In [ ]:
ver = (COV.groupby("app_version", dropna=False)
          .agg(rows_all=("rows_all","sum"), rows_with_path=("rows_with_path","sum"),
               installs_all=("installs_all","sum"), installs_with_path=("installs_with_path","sum"))
          .reset_index())
ver["rows_path_%"]     = (100*ver.rows_with_path/ver.rows_all).round(1)
ver["installs_path_%"] = (100*ver.installs_with_path/ver.installs_all).round(1)
ver = ver.sort_values("installs_all", ascending=False)
print("Coverage of installed_path by app version — the gate is the client, not the data:")
display(ver.reset_index(drop=True))

capable = ver[ver["installs_path_%"] > 50]
tot_rows, tot_inst = ver.rows_all.sum(), ver.installs_all.sum()
cap_rows, cap_inst = capable.rows_all.sum(), capable.installs_all.sum()
print(f"\nVersions that report a path: {sorted(capable.app_version.dropna().tolist())}")
print(f"Judgeable subset: {cap_inst:,}/{tot_inst:,} installs ({100*cap_inst/tot_inst:.0f}%) "
      f"and {cap_rows:,}/{tot_rows:,} measurements ({100*cap_rows/tot_rows:.0f}%).")
print("Everything below describes THAT subset. Older clients are not un-scoped — they are unobserved.")

# per-country capability: where is the fleet too old to audit at all?
cc = (COV.groupby("iso3").agg(installs_all=("installs_all","sum"),
                              installs_with_path=("installs_with_path","sum")).reset_index())
cc["judgeable_%"] = (100*cc.installs_with_path/cc.installs_all).round(1)
cc = cc[cc.installs_all >= MIN_INSTALLS].sort_values("judgeable_%")
print(f"\nLeast-auditable deployments (countries with >= {MIN_INSTALLS} installs) — "
      "a low share here is an upgrade problem, not an install-scope problem:")
display(cc.head(10).reset_index(drop=True))

## §3 · Fleet view — how the choice was actually made

One row per country: the split of its 2.0.2+ installs across the three path families.
`other` is the tell-tale of an unsupported deployment — the app unpacked to a Desktop or
a second drive rather than installed.

In [ ]:
def scope_mix(df, key):
    g = (df.pivot_table(index=key, columns="scope", values="browser_id",
                        aggfunc="count", fill_value=0))
    for s in SCOPE_ORDER:
        if s not in g.columns: g[s] = 0
    g = g[SCOPE_ORDER]
    g["installs"] = g.sum(axis=1)
    for s in SCOPE_ORDER:
        g[f"{s}_%"] = (100*g[s]/g["installs"]).round(1)
    return g.reset_index()

cty = scope_mix(INST, "iso3")
cty["schools"] = INST.groupby("iso3").school_id.nunique().reindex(cty.iso3).values
cty["name"]    = cty.iso3.map(ISO3_NAME).fillna(cty.iso3)
cty = cty[cty.installs >= MIN_INSTALLS].sort_values("all_users_%")

f = INST.scope.value_counts()
print(f"FLEET · {len(INST):,} installs: "
      + " · ".join(f"{s} {f.get(s,0):,} ({100*f.get(s,0)/len(INST):.0f}%)" for s in SCOPE_ORDER))
print(f"\nBy country (>= {MIN_INSTALLS} installs), worst compliance first:")
display(cty[["iso3","name","schools","installs","all_users_%","per_user_%","other_%"]]
        .reset_index(drop=True))

top = cty.nlargest(20, "installs").sort_values("all_users_%")
fig, ax = plt.subplots(figsize=(10, max(4, 0.34*len(top))))
left = np.zeros(len(top))
for s in SCOPE_ORDER:
    ax.barh(top.name, top[f"{s}_%"], left=left, color=SCOPE_COLOR[s],
            label=s.replace("_"," "), height=0.72)
    left += top[f"{s}_%"].values
ax.axvline(50, color=GIGA_GREY[700], lw=0.8, ls=":")
ax.set_xlim(0, 100); ax.set_xlabel("share of installs (%)")
ax.set_title("Install scope by country — the 20 largest deployments")
ax.legend(loc="lower right", frameon=False, ncol=3, fontsize=9)
plt.tight_layout(); plt.show()

## §4 · Does the choice change anything? — accounts measuring per machine

The point of an all-users install is that a second teacher logging into the same PC keeps
measuring. That is testable: group installs by `device_hardware_id` and count the distinct
Windows accounts that produced a measurement.

**Validity check first.** The test only means something if `windows_username` is captured
per measurement rather than frozen at install time — otherwise a one-account-per-machine
result would be an artefact of the field, not a fact about schools. The cell checks this
before reporting anything.

**Denominator warning.** `device_hardware_id` is deliberately scrubbed on low-end hardware
(Uzbekistan, Malawi), so this section covers only machines that report one. Devices whose
installs disagree about scope are excluded rather than assigned to either side.

In [ ]:
# ── validity check: does windows_username actually vary within one install? ──
varies = (INST.n_usernames > 1).sum()
print(f"VALIDITY · installs whose windows_username changes over their life: {varies:,} "
      f"(max {INST.n_usernames.max()} accounts on one install).")
print("  → the field is a per-measurement capture of the logged-in account, not a value\n"
      "    frozen at install time. A low multi-account rate below is therefore a fact about\n"
      "    how schools use their PCs, not an artefact of the column.\n")

covered = INST.device_hardware_id.notna().sum()
mixed   = DEV[DEV.n_scopes > 1]
dev     = DEV[DEV.n_scopes == 1]

print(f"Installs on a machine that reports a hardware id: {covered:,}/{len(INST):,} "
      f"({100*covered/len(INST):.0f}%) → {len(DEV):,} machines "
      f"({len(mixed):,} excluded as scope-mixed).")
print("Countries under-represented here (hardware id scrubbed):")
hw = (INST.assign(has_hw=INST.device_hardware_id.notna())
        .groupby("iso3").has_hw.agg(["sum","count"]))
hw["hw_%"] = (100*hw["sum"]/hw["count"]).round(1)
display(hw[hw["count"] >= MIN_INSTALLS].sort_values("hw_%").head(5)
          .rename(columns={"sum":"with_hw_id","count":"installs"}))

out = (dev.groupby("scope")
         .agg(machines=("device_hardware_id","count"), mean_accounts=("n_accounts","mean"),
              multi_account=("n_accounts", lambda s: (s > 1).sum()),
              med_tests=("n_tests","median"))
         .reset_index())
out["multi_account_%"] = (100*out.multi_account/out.machines).round(2)
out["mean_accounts"]   = out.mean_accounts.round(3)
print("\nDo all-users machines see more than one Windows account measuring?")
display(out[["scope","machines","mean_accounts","multi_account","multi_account_%","med_tests"]])

a = out.set_index("scope")
if {"all_users","per_user"} <= set(a.index):
    au, pu_ = a.loc["all_users","multi_account_%"], a.loc["per_user","multi_account_%"]
    print(f"\nAll-users machines are {au/max(pu_,1e-9):.1f}x likelier to see a second account "
          f"measuring ({au:.1f}% vs {pu_:.1f}%) — but the absolute rate is {au:.1f}%.")
    print("Read that carefully: the permission is necessary, not sufficient. Choosing all-users\n"
          "unblocks a second account; it does not create one. Almost every school PC in the fleet\n"
          "measures from exactly one Windows login either way — so the gain from fixing the\n"
          "installer setting is resilience when the account changes, not more measurement today.")

# ── why the rate is so low: the fleet logs in under ONE shared account ──
GENERIC_LOGIN = {"user","admin","1","teacher","administrator","dell","hp","win10","pc","server",
                 "student","lenovo","acer","asus","new","user1","professional","korisnik",
                 "owner","default","home","admins"}
u = INST.windows_username.fillna("").str.lower()
top5 = u.value_counts().head(5)
print(f"\nWhy the rate is low — the fleet is not running per-teacher logins:")
print(f"  {u.isin(GENERIC_LOGIN).sum():,} installs ({100*u.isin(GENERIC_LOGIN).mean():.0f}%) "
      f"run under a generic shared account name;")
print(f"  the five commonest logins cover {100*top5.sum()/len(INST):.0f}% of all installs "
      f"({u.nunique():,} distinct names in total).")
display(top5.rename("installs").to_frame())
print("A shared PC in these schools is shared through ONE Windows login, not through a login\n"
      "per teacher. That is the reason the installer setting has little measurable effect today:\n"
      "there is usually no second account for an all-users install to serve. The setting becomes\n"
      "material only where schools move to per-person or managed profiles.\n")

multi = dev[dev.n_accounts > 1]
print(f"\nMachines where 2+ accounts really do measure: {len(multi):,} "
      f"({100*len(multi)/len(dev):.2f}% of machines), carrying {multi.n_tests.sum():,} measurements")
display(multi.sort_values("n_accounts", ascending=False)
          .head(10)[["iso3","scope","n_accounts","n_installs","n_schools","n_tests"]]
          .reset_index(drop=True))

## §5 · Whose profile is it? — generic, vendor and technician accounts

A per-user install is only as durable as the Windows account it sits in. The account name
is embedded in the path, so the failure modes are visible:

* **generic box accounts** (`user`, `admin`, `administrator`, `pc`, `dell`, `hp`) — the
  factory profile nobody owns;
* **vendor / technician accounts** — one company name appearing across many *different*
  schools (`MULTI_ACCOUNT` threshold). The installer created a profile for themselves and
  installed into it. When the school stops using that login, measurement stops, and no
  amount of re-briefing the teacher will restart it.

In [ ]:
pu = INST[INST.scope == "per_user"].copy()
acc = (pu.groupby("path_user")
         .agg(installs=("browser_id","nunique"), schools=("school_id","nunique"),
              countries=("iso3","nunique"), tests=("n_tests","sum"),
              med_days=("days_measured","median")).reset_index()
         .sort_values("installs", ascending=False))

GENERIC = {"user","users","admin","admins","administrator","администратор","админ","pc","dell","hp",
           "lenovo","acer","asus","toshiba","sony","samsung","win10","win11","windows","home",
           "owner","student","teacher","учитель","o'qituvchi","o'quvchi","server","new","1","user1",
           "professional","korisnik","default"}
acc["kind"] = np.where(acc.path_user.isin(GENERIC), "generic",
              np.where(acc.schools >= MULTI_ACCOUNT, "shared/vendor", "named"))

print(f"Per-user installs: {len(pu):,} across {acc.path_user.nunique():,} distinct account names.")
summ = (acc.groupby("kind").agg(account_names=("path_user","count"), installs=("installs","sum"),
                                schools=("schools","sum")).reset_index())
summ["installs_%"] = (100*summ.installs/summ.installs.sum()).round(1)
display(summ)

print(f"\nAccount names on >= {MULTI_ACCOUNT} schools — shared images and installer-owned profiles:")
display(acc[acc.schools >= MULTI_ACCOUNT].head(25).reset_index(drop=True))

vendor = acc[(acc.kind == "shared/vendor") & (~acc.path_user.isin(GENERIC))]
print(f"\nNon-generic names spanning many schools = {int(vendor.installs.sum()):,} installs "
      f"at {int(vendor.schools.sum()):,} school-slots, concentrated in "
      f"{vendor.countries.max() if len(vendor) else 0} or fewer countries each.")
print("These are the highest-value remediation targets: one conversation with one contractor\n"
      "converts hundreds of installs, and each one is currently a single point of failure.")

# cross-profile oddity: the app runs while a different account is logged in
cross = pu[pu.windows_username.notna() &
           (pu.windows_username.str.lower() != pu.path_user.fillna(""))]
print(f"\nPer-user installs measuring under a DIFFERENT logged-in account than the path owner: "
      f"{len(cross):,} ({100*len(cross)/max(len(pu),1):.1f}%) — a per-user install reaching a\n"
      "second account anyway (roaming profile, renamed account, or a shared image).")

## §6 · Is it improving? — scope by install cohort

Two things live here, and they must not be confused.

**The trend** — the share of new installs that chose all-users, by the month the install
first reported. This is the only honest read of whether deployment guidance is landing.

**The trap** — comparing all-users vs per-user *lifetimes* across the whole window makes
per-user installs look far healthier. They are not: they are simply older. Controlled
within a cohort the difference collapses. Anything that ranks scopes by retention without
this control is reporting the age of the fleet.

In [ ]:
stable = INST[INST.n_scopes == 1]
coh = (stable.pivot_table(index="cohort", columns="scope", values="browser_id",
                          aggfunc="count", fill_value=0))
for s in SCOPE_ORDER:
    if s not in coh.columns: coh[s] = 0
coh = coh[SCOPE_ORDER]
coh["installs"]    = coh.sum(axis=1)
coh["all_users_%"] = (100*coh.all_users/coh.installs).round(1)
med = stable.pivot_table(index="cohort", columns="scope", values="days_measured", aggfunc="median")
coh = coh.join(med.rename(columns={s: f"med_days_{s}" for s in med.columns}))
display(coh.reset_index())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
ax1.bar(coh.index, coh.installs, width=22, color=GIGA_GREY[300], label="installs")
ax1b = ax1.twinx()
ax1b.plot(coh.index, coh["all_users_%"], color=GIGA_PRIMARY[600], marker="o", lw=2)
ax1b.set_ylim(0, 100); ax1b.set_ylabel("all-users share (%)", color=GIGA_PRIMARY[600])
ax1.set_ylabel("new installs"); ax1.set_title("Compliance of new installs, by cohort")
for s in ("all_users","per_user"):
    if f"med_days_{s}" in coh:
        ax2.plot(coh.index, coh[f"med_days_{s}"], marker="o", lw=2,
                 color=SCOPE_COLOR[s], label=s.replace("_"," "))
ax2.set_ylabel("median days measured"); ax2.legend(frameon=False)
ax2.set_title("Lifetime within cohort — the lines sit on top of each other")
for a in (ax1, ax2): a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

naive = stable.groupby("scope").days_measured.median()
print("Naive (pooled across the window) median days measured:")
print(naive.to_string())
print("\nSame comparison, computed within each cohort and then averaged:")
inside = (coh[[c for c in coh.columns if c.startswith("med_days_")]]
          .mean().rename(lambda c: c.replace("med_days_","")))
print(inside.round(1).to_string())
print("\nIf the pooled ordering flips or shrinks against the within-cohort one, the pooled\n"
      "number was measuring fleet age. Quote the within-cohort figure.")

## §7 · Country drill-down — the remediation list

Which schools in one country are running on a per-user or unsupported install, and which
Windows account each one depends on. This is the exportable artefact: a per-school list a
country team can act on without rerunning anything.

In [ ]:
focus = COUNTRY or cty.nlargest(1, "installs").iloc[0]["iso3"]
F = INST[INST.iso3 == focus].copy()
print(f"FOCUS: {focus} ({ISO3_NAME.get(focus, focus)}) · {len(F):,} installs "
      f"· {F.school_id.nunique():,} schools · {F.n_tests.sum():,} measurements\n")

sch = scope_mix(F, "school_id")
sch["installs_ok"] = sch["all_users"]
sch["status"] = np.where(sch.all_users > 0,
                  np.where(sch.per_user + sch.other > 0, "mixed", "compliant"), "at_risk")
st = sch.status.value_counts()
for k in ("compliant","mixed","at_risk"):
    print(f"  {k:<10} {st.get(k,0):>6,} schools ({100*st.get(k,0)/len(sch):.1f}%)")
print("\n'at_risk' = every install at that school is per-user or unsupported: if that one\n"
      "Windows account stops being used, the school goes dark and looks like an outage.")

risk = (F[F.scope != "all_users"]
        .merge(sch[["school_id","status"]], on="school_id")
        .query("status == 'at_risk'")
        .groupby(["school_id","giga_id_school"])
        .agg(installs=("browser_id","nunique"),
             accounts=("path_user", lambda s: ", ".join(sorted(set(s.dropna())))),
             app_version=("app_version","max"), tests=("n_tests","sum"),
             days=("days_measured","max"), last_seen=("last_seen","max"))
        .reset_index().sort_values("tests", ascending=False))
print(f"\nAt-risk schools in {focus} — {len(risk):,} rows, heaviest measurement volume first:")
display(risk.head(20).reset_index(drop=True))

## §8 · What this means

1. **Report the judgeable subset, never the whole fleet.** `installed_path` starts at app
   2.0.2; a country with an old fleet has an *upgrade* problem that looks like a
   compliance problem. Quote §2's denominator next to any compliance figure.
2. **The compliance number is real and it is moving.** The share of new installs choosing
   all-users is the one metric worth putting in a deployment report — per cohort, not
   pooled.
3. **All-users is necessary, not sufficient.** It unblocks a second Windows account; it
   does not produce one. Almost every machine in the fleet measures from a single login
   regardless of scope, so do not promise coverage gains from the installer setting alone
   — the gain is resilience, not volume.
4. **Per-user installs under a vendor or generic account are the fragile ones.** They are
   a single point of failure with a name attached, they cluster by contractor, and §5
   exports exactly who to talk to.
5. **Never compare scopes pooled across the window.** All-users installs are younger;
   pooled lifetime comparisons measure fleet age and will say the wrong thing.
6. **Two things the data cannot tell us** — whether a per-user install was a deliberate
   choice or the wizard default being clicked through, and whether a machine has other
   Windows accounts that simply never log in. Both need a field check, not a query.

In [ ]:
cty.to_csv(OUT / "installscope_by_country.csv", index=False)
acc.to_csv(OUT / "installscope_accounts.csv", index=False)
coh.reset_index().to_csv(OUT / "installscope_by_cohort.csv", index=False)
risk.to_csv(OUT / f"installscope_at_risk_{focus}.csv", index=False)
sch.to_csv(OUT / f"installscope_schools_{focus}.csv", index=False)
print("exported →",
      "\n  outputs/installscope_by_country.csv",
      "\n  outputs/installscope_accounts.csv",
      "\n  outputs/installscope_by_cohort.csv",
      f"\n  outputs/installscope_at_risk_{focus}.csv",
      f"\n  outputs/installscope_schools_{focus}.csv")